<a href="https://colab.research.google.com/github/baxodirmamatvaliev-del/JaydariGPT/blob/main/Jaydari_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformers bitsandbytes datasets peft accelerate trl "torchao>=0.16.0"

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from trl import SFTTrainer

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # DATA
tokenizer = AutoTokenizer.from_pretrained(model_id)

print('Vocab size:', tokenizer.vocab_size)
print('Special tokens:', tokenizer.special_tokens_map)


# Quantization - MODUL xotirasini ozxolida kichikroq saqlash uchun
# m: 4K --> 1080p
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)



In [ ]:
bnb_config

model = AutoModelForCausalLM.from_pretrained( # llama ni data sini yuklab oldik
    model_id,  # llamani topish
    quantization_config=bnb_config,
    device_map="auto", # gpu / cpu
    dtype=torch.float16
)

In [ ]:
# Before Fine-Tuning
prompt = "Explain what a tokenizer is."

#Savolni PyTorch tensorlari — token raqamlariga aylantiradi.
inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

print(model)
print("======================")
print(model.config)


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")


Datasets library
# Instruction tuning

In [ ]:
from datasets import load_dataset

dataset = load_dataset('yahma/alpaca-cleaned', split='train')
dataset

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}\n\n"
    )

generate_prompt(dataset[1])

In [ ]:
def formatting_func(example):
    return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [ ]:
dataset[0]['text'] #yangi future

In [ ]:
dataset = dataset.select(range(7000)) #memoriylar tugap qolmasligi uchun 7000 data ni ozida instraction tuning qilamz

In [ ]:
dataset = dataset.shuffle(seed=42) # endi random holatda 7000 malumotni ol / seed=42 malumotlarni ertaga ham bir xil saqlash

In [ ]:
dataset

In [ ]:
# Full Fine-tuning  =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,  # rank qancha kichik bolsa shuncha kichik parametrni tuning qiladi bu memoryga gpuga yaxshi
    lora_alpha=16,  #scaling qilib beradi , yani tozalash
    lora_dropout=0.05, #overfit bolishini oldini olish uchun
    task_type="CAUSAL_LM",	# Vazifa — oldingi tokenlardan keyingi tokenni bashorat qilib matn yaratish.
    target_modules=["q_proj", "v_proj"], #bular model attention qismida layeridan olinmoqda va bu ikkita layer eng muhim qilsmlari hisoblanadi
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer


training_args = SFTConfig(
    output_dir="./jaydari_gpt", #Natijalar saqlanadigan papka.
    per_device_train_batch_size=1, # Har bir GPU bir safar 1 ta misolni qayta ishlaydi.
    gradient_accumulation_steps=4, # VRAM kamroq ishlatish uchun
    num_train_epochs=2,  # train 2 martda
    logging_steps=20, # Har 20 ta parametr yangilash qadamida loss kabi ko‘rsatkichlarni chiqaradi.
    save_strategy="epoch", # har bir epoch save qiladi va train jarayoinida xatolik chiqsa oldingi save epoch ni..
    fp16=True,
    bf16=False,
    loss_type="nll",
    max_length=512,
    report_to="none" # kuzatuv  loglarni xizmatlariga hisobot yubormaydi.
)



In [ ]:
print(dataset.column_names)

In [ ]:
import importlib

# Disable unsupported CUDA mixed-dtype operation on T4
chunked_logprob = importlib.import_module(
    "trl.kernels.chunked_logprob"
)

chunked_logprob._MM_OUT_DTYPE = False

print("T4 compatibility workaround enabled")

In [ ]:
# T4: keep trainable LoRA parameters in float32 for FP16 GradScaler.
model.zero_grad(set_to_none=True)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

# Check after trainer initialization, which can change adapter dtypes.
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
print("Trainable dtypes:", {p.dtype for p in model.parameters() if p.requires_grad})

trainer.train()
model.save_pretrained("jaydari_gpt")
tokenizer.save_pretrained("jaydari_gpt")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM # base model yuklab olyapmz
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)

model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()


In [ ]:
prompt = "Explain what machine learning is in simple words." ## oddiy savol ozini qaytardi

inputs_base = tokenizer_base(
    prompt,
    return_tensors="pt"
).to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

# FINE-TUNED MODEL INFERENCE

In [ ]:
model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

In [ ]:
prompt = "### Instruction:\nExplain what machine learning is in simple words.\n\n### Response:\n"

inputs_base = tokenizer_base(
    prompt,
    return_tensors="pt"
).to(model_base.device)

with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

In [ ]:
!zip -r jaydari_gpt.zip jaydari_gpt

In [ ]:
from google.colab import  files
files.download("jaydari_gpt.zip")

In [5]:
!pip install -q langdetect #

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# MODEL TESTING

In [1]:
!unzip jaydari_gpt.zip

Archive:  jaydari_gpt.zip
   creating: jaydari_gpt/
   creating: jaydari_gpt/checkpoint-3500/
  inflating: jaydari_gpt/checkpoint-3500/trainer_state.json  
  inflating: jaydari_gpt/checkpoint-3500/training_args.bin  
  inflating: jaydari_gpt/checkpoint-3500/adapter_model.safetensors  
  inflating: jaydari_gpt/checkpoint-3500/scaler.pt  
  inflating: jaydari_gpt/checkpoint-3500/rng_state.pth  
  inflating: jaydari_gpt/checkpoint-3500/adapter_config.json  
  inflating: jaydari_gpt/checkpoint-3500/chat_template.jinja  
  inflating: jaydari_gpt/checkpoint-3500/optimizer.pt  
  inflating: jaydari_gpt/checkpoint-3500/README.md  
  inflating: jaydari_gpt/checkpoint-3500/tokenizer_config.json  
  inflating: jaydari_gpt/checkpoint-3500/scheduler.pt  
  inflating: jaydari_gpt/checkpoint-3500/tokenizer.json  
  inflating: jaydari_gpt/adapter_model.safetensors  
  inflating: jaydari_gpt/adapter_config.json  
  inflating: jaydari_gpt/chat_template.jinja  
  inflating: jaydari_gpt/README.md   
  inf

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM # class larni chaqirib oldik
from peft import PeftModel, PeftConfig
%pip install -U "torchao>=0.16.0"


base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [10]:
from langdetect import detect_langs

def ask_jaydari_gpt(prompt: str, max_new_tokens=256):
    try:
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Block only if we're confident it's NOT English
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
        return "Sorry, you should ask a question only in English."

    # Proceed with generation
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()


In [11]:
ask_jaydari_gpt('what is ML')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🌍 Detected lang: en (1.00)


'Machine Learning (ML) is a field of artificial intelligence that is used to develop systems that can learn from data and make predictions or decisions based on it. It is a subset of AI that focuses on developing algorithms that can analyze and interpret large amounts of data to make accurate predictions or decisions.'